# 7. De caràcters a significat: Embeddings i Cerca Semàntica

Tractament de Dades Textuals i Codificades — Eixample Clínic, 2026

Pol Pastells, ppastells@eixampleclinic.es

**Objectius de la sessió:**
- Entendre el camí de caràcters → tokens → embeddings
- Aprendre a generar embeddings amb `sentence-transformers`
- Comparar textos per significat amb similitud cosinus
- Construir un cercador semàntic sobre dades mèdiques reals

# Part 0 — De n-grames de caràcters a embeddings

A la pràctica anterior vam representar el llenguatge caràcter a caràcter. Avui farem el salt cap a com ho fan els models moderns.

```
Caràcters → Paraules → Subparaules (tokens) → Vectors (embeddings)
```

## On érem: n-grames de caràcters

Recordeu que vam fer un model que aprenia seqüències de caràcters:

```
"p-a-c-i-e-n-t" → el caràcter següent més probable
```

**Avantatges**: molt simple, funciona amb qualsevol idioma, no cal decidir què és una "paraula".

**Limitació fonamental**: el model no té cap noció de *significat*. Només sap quines lletres solen anar juntes. No "sap" que "diabetis" i "diabetes" estan relacionades.

## Primer intent: tokenitzar per paraules

La idea més intuïtiva seria treballar amb paraules senceres:

```python
"El pacient té diabetis" → ["El", "pacient", "té", "diabetis"]
```

### Problemes

| Problema | Exemple |
|----------|---------|  
| **Vocabulari enorme** | En anglès mèdic hi ha centenars de milers de termes |
| **Paraules noves** | "COVID-19" no existia al 2019, "SARS-CoV-2" tampoc |
| **Variacions** | "hipertensió", "hipertensiva", "hipertens" → 3 entrades diferents |
| **Errors ortogràfics** | "diabetis" vs "diabètis" → paraula desconeguda |

## La solució moderna: subparaules (subword tokenization)

En lloc de caràcters o paraules senceres, els models moderns tallen en **trossos intermitjos**:

```
"hyperglycemia" → ["hyper", "##glyc", "##emia"]
"hypoglycemia"  → ["hypo",  "##glyc", "##emia"]
"hypernatremia" → ["hyper", "##natr", "##emia"]
```

El model aprèn que:
- `"hyper"` = excés de
- `"hypo"` = deficiència de  
- `"##emia"` = relacionat amb la sang
- `"##glyc"` = relacionat amb glucosa

Això és molt semblant a com funciona l'etimologia mèdica! Els prefixos i sufixos porten significat.

### Per què és útil?

- **Vocabulari manejable**: ~30.000-100.000 peces en lloc de milions de paraules
- **Paraules noves**: es poden descompondre en peces conegudes
- **Comparteixen significat**: "cardio-" contribueix el mateix a "cardiologia" i "cardiovascular"

## De tokens a vectors: Embeddings

Un cop tenim els tokens, necessitem convertir-los en **nombres** que el model pugui processar.

### Opció simple: One-hot encoding
Cada token és un vector amb un 1 i la resta 0s:
```
"hyper" → [0, 0, 1, 0, 0, ..., 0]  (posició 2)
"hypo"  → [0, 0, 0, 1, 0, ..., 0]  (posició 3)
```

**Problema**: "hyper" i "hypo" estan igual de lluny que "hyper" i "pizza". No hi ha noció de similitud.

### Solució: Embeddings
Cada token es representa com un **vector dens** de dimensions fixes (per exemple, 384 nombres):
```
"hyper" → [0.23, -0.45, 0.12, ..., 0.89]  (384 dimensions)
"hypo"  → [0.21, -0.43, 0.15, ..., 0.87]  (similar!)
"pizza" → [-0.82, 0.31, -0.56, ..., 0.02] (molt diferent)
```

Tokens amb significat similar tenen vectors **propers**.

## El pipeline complet d'un transformer

```
Text original: "El pacient presenta hiperglucèmia"
  │
  ▼  Tokenització
Tokens: ["El", "paci", "##ent", "presenta", "hiper", "##gluc", "##èmia"]
  │
  ▼  Taula d'embeddings
Vectors inicials: 7 vectors de 384 dimensions
  │
  ▼  Transformer (atenció entre tokens → context)
Vectors contextualitzats: 7 vectors de 384 dims (ara cada vector
                          incorpora informació de tota la frase)
  │
  ▼  Pooling (mitjana)
Embedding de frase: 1 vector de 384 dims
  │
  ▼  Tasca final
Similitud, classificació, NER, QA...
```

### Embeddings contextualitzats

La mateixa paraula pot tenir vectors **diferents** segons el context:

```
"El pacient té la tensió alta"    → "tensió" ≈ pressió arterial
"Hi ha molta tensió a l'equip"   → "tensió" ≈ estrès
```

Els vectors seran DIFERENTS perquè el model mira tota la frase.

## Resum vs la pràctica anterior

| | Pràctica anterior | Avui |
|---|---|---|
| **Unitat bàsica** | Caràcters | Subparaules (tokens) |
| **Representació** | Un sol número per lletra | Vector de 384 dims per token |
| **Què aprèn** | Patrons de lletres | Significat semàntic |
| **Context** | N caràcters anteriors | Tota la frase |
| **Capacitat** | Genera seqüències | Entén i compara significats |

---

# Part 1 — Primers embeddings amb sentence-transformers

El paquet `sentence-transformers` ens permet generar embeddings de frases senceres amb una sola línia de codi.

In [ ]:
# Instal·lació
# !pip install sentence-transformers datasets

> ⚠️ Si veus un warning sobre `position_ids UNEXPECTED`, és normal i pots ignorar-lo.
> Apareix perquè el model es va guardar amb una versió anterior de la llibreria.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Carreguem un model lleuger però bo
# 'all-MiniLM-L6-v2' genera vectors de 384 dimensions
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model carregat!")

In [ ]:
# Generem l'embedding d'una frase
frase = "The patient presents with acute chest pain"
embedding = model.encode(frase)

print(f"Frase: '{frase}'")
print(f"Tipus: {type(embedding)}")
print(f"Dimensions: {embedding.shape}")
print(f"Primers 10 valors: {embedding[:10].round(3)}")

In [ ]:
# Podem generar embeddings de múltiples frases alhora
frases = [
    "The patient has type 2 diabetes mellitus",
    "Diagnosed with diabetes, insulin-dependent",
    "The patient fractured the left femur",
    "Broken leg after a fall",
]

embeddings = model.encode(frases)
print(f"Matriu d'embeddings: {embeddings.shape}")
print(f"→ {len(frases)} frases, cadascuna representada per {embeddings.shape[1]} nombres")

# Part 2 — Similitud cosinus: comparar significats

Dos vectors són similars si apunten en la mateixa direcció. La **similitud cosinus** mesura exactament això:

$$\text{similitud}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

- **1.0** = idèntics en significat
- **0.0** = no relacionats
- **-1.0** = oposats (rar en la pràctica)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Comparem les 4 frases entre elles
sim_matrix = cosine_similarity(embeddings)

# Visualitzem
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)

# Etiquetes curtes
etiquetes = [f[:30] + "..." if len(f) > 30 else f for f in frases]
ax.set_xticks(range(len(frases)))
ax.set_yticks(range(len(frases)))
ax.set_xticklabels(etiquetes, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(etiquetes, fontsize=9)

# Valors dins les cel·les
for i in range(len(frases)):
    for j in range(len(frases)):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center", fontsize=11)

plt.colorbar(im, label="Similitud cosinus")
plt.title("Matriu de similitud semàntica")
plt.tight_layout()
plt.show()

### Observa:
- Les frases sobre **diabetis** (0 i 1) tenen alta similitud entre elles
- Les frases sobre **fractures** (2 i 3) també
- La similitud **entre temes** és molt més baixa
- Nota que les frases usen **paraules molt diferents** però el model capta el significat!

### Provem amb un exemple clínic més ric

In [ ]:
frases_cliniques = [
    # Grup 1: Dolor toràcic / cardíac
    "Patient complains of severe chest pain radiating to left arm",
    "Acute myocardial infarction suspected",
    "ECG shows ST elevation in leads II, III, aVF",
    # Grup 2: Problemes respiratoris
    "Shortness of breath with productive cough",
    "Bilateral pulmonary infiltrates on chest X-ray",
    "Patient requires supplemental oxygen therapy",
    # Grup 3: Diabetis
    "Blood glucose level is 350 mg/dL",
    "Patient with poorly controlled type 2 diabetes",
    "HbA1c is 11.2%, indicating chronic hyperglycemia",
]

emb_clinics = model.encode(frases_cliniques)
sim_clinics = cosine_similarity(emb_clinics)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_clinics, cmap="RdYlGn", vmin=0, vmax=1)

etiquetes = [f[:45] + "..." if len(f) > 45 else f for f in frases_cliniques]
ax.set_xticks(range(len(frases_cliniques)))
ax.set_yticks(range(len(frases_cliniques)))
ax.set_xticklabels(etiquetes, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(etiquetes, fontsize=8)

for i in range(len(frases_cliniques)):
    for j in range(len(frases_cliniques)):
        ax.text(j, i, f"{sim_clinics[i,j]:.2f}", ha="center", va="center", fontsize=8)

# Línies per separar grups
for pos in [2.5, 5.5]:
    ax.axhline(y=pos, color="black", linewidth=2)
    ax.axvline(x=pos, color="black", linewidth=2)

plt.colorbar(im, label="Similitud cosinus")
plt.title("Similitud semàntica — Frases clíniques agrupades per tema")
plt.tight_layout()
plt.show()

### Preguntes

Mira la matriu: 
- Es formen els 3 grups clarament?
- Hi ha alguna frase que queda "entre dos grups"? Per què pot ser?
- La frase de l'ECG (que és molt tècnica), queda ben agrupada amb les de dolor toràcic?

# Part 3 — Dataset real: Medical Questions Pairs

Ara treballarem amb dades reals: un dataset de ~3.000 parells de preguntes mèdiques amb una etiqueta que indica si són equivalents.

In [ ]:
from datasets import load_dataset

ds = load_dataset("medical_questions_pairs", split="train")
print(f"Nombre de parells: {len(ds)}")
print(f"Columnes: {ds.column_names}")
print()

In [ ]:
ds

In [ ]:
ds[0]

In [ ]:
# Mirem uns quants exemples
for i in range(5):
    ex = ds[i]
    equiv = "✅ Similars" if ex["label"] == 1 else "❌ No similars"
    print(f"{equiv}")
    print(f"  Q1: {ex['question_1']}")
    print(f"  Q2: {ex['question_2']}")
    print()

In [ ]:
df = ds.to_pandas()
print("Distribució d'etiquetes:")
print(df["label"].value_counts())
print(f"\nTotal preguntes úniques (Q1): {df['question_1'].nunique()}")
print(f"Total preguntes úniques (Q2): {df['question_2'].nunique()}")

## 3.1 — Generem embeddings de totes les preguntes

Crearem un corpus amb totes les preguntes úniques i els seus embeddings.

In [ ]:
# Recollim totes les preguntes úniques
totes_preguntes = list(set(df["question_1"].tolist() + df["question_2"].tolist()))
print(f"Total preguntes úniques: {len(totes_preguntes)}")
print("\nExemples:")
for q in totes_preguntes[:5]:
    print(f"  - {q}")

In [ ]:
# Generem els embeddings (pot trigar ~1-2 minuts)
print("Generant embeddings... (pot trigar una mica)")
embeddings_corpus = model.encode(totes_preguntes, show_progress_bar=True, batch_size=64)
print(f"\nMatriu d'embeddings: {embeddings_corpus.shape}")
print(f"→ {embeddings_corpus.shape[0]} preguntes × {embeddings_corpus.shape[1]} dimensions")

# Part 4 — Cercador semàntic

Ara construirem un **cercador**: donada una pregunta nova, trobarem les preguntes més similars del corpus.

In [ ]:
def cercar(query, corpus_textos, corpus_embeddings, model, top_k=5):
    """
    Cerca semàntica: troba els textos més similars a la query.

    Args:
        query: text a cercar
        corpus_textos: llista de textos del corpus
        corpus_embeddings: matriu d'embeddings del corpus
        model: model de sentence-transformers
        top_k: nombre de resultats a retornar

    Returns:
        Llista de tuples (text, similitud)
    """
    # 1. Generem l'embedding de la query
    query_emb = model.encode([query])

    # 2. Calculem similitud amb tot el corpus
    similituds = cosine_similarity(query_emb, corpus_embeddings)[0]

    # 3. Ordenem per similitud (descendent)
    top_indices = np.argsort(similituds)[::-1][:top_k]

    # 4. Retornem resultats
    resultats = []
    for idx in top_indices:
        resultats.append((corpus_textos[idx], similituds[idx]))

    return resultats

In [ ]:
def mostrar_resultats(query, resultats):
    """Mostra els resultats de forma neta."""
    print(f'🔍 Query: "{query}"')
    print(f"{'='*70}")
    for i, (text, sim) in enumerate(resultats):
        barra = "█" * int(sim * 30)
        print(f"  {i+1}. [{sim:.3f}] {barra}")
        print(f"     {text}")
        print()

In [ ]:
# Provem el cercador
query = "What causes headaches?"
resultats = cercar(query, totes_preguntes, embeddings_corpus, model, top_k=5)
mostrar_resultats(query, resultats)

In [ ]:
# Prova amb una pregunta sobre diabetis
query = "How to manage high blood sugar levels?"
resultats = cercar(query, totes_preguntes, embeddings_corpus, model, top_k=5)
mostrar_resultats(query, resultats)

In [ ]:
# Prova amb vocabulari molt diferent però significat similar
query = "My heart is beating very fast and I feel dizzy"
resultats = cercar(query, totes_preguntes, embeddings_corpus, model, top_k=5)
mostrar_resultats(query, resultats)

In [ ]:
# Prova amb una pregunta que NO tingui res a veure amb medicina
query = "What is the best programming language for beginners?"
resultats = cercar(query, totes_preguntes, embeddings_corpus, model, top_k=5)
mostrar_resultats(query, resultats)

print("👆 Nota com les similituds són molt més baixes que abans!")

# Part 5 — Validació: els embeddings detecten equivalència?

El dataset té etiquetes de parells equivalents. Podem comprovar si la similitud cosinus captura bé l'equivalència.

In [ ]:
# Calculem la similitud per cada parell del dataset
print("Calculant similituds per tots els parells...")

emb_q1 = model.encode(df["question_1"].tolist(), show_progress_bar=True, batch_size=64)
emb_q2 = model.encode(df["question_2"].tolist(), show_progress_bar=True, batch_size=64)

# Similitud cosinus element a element
similituds_parells = np.array([cosine_similarity([e1], [e2])[0][0] for e1, e2 in zip(emb_q1, emb_q2)])

df["similitud"] = similituds_parells

In [ ]:
# Visualitzem la distribució de similituds per cada grup
fig, ax = plt.subplots(figsize=(10, 5))

equiv = df[df["label"] == 1]["similitud"]
no_equiv = df[df["label"] == 0]["similitud"]

ax.hist(no_equiv, bins=50, alpha=0.6, label=f"No equivalents (n={len(no_equiv)})", color="red")
ax.hist(equiv, bins=50, alpha=0.6, label=f"Equivalents (n={len(equiv)})", color="green")

ax.set_xlabel("Similitud cosinus", fontsize=12)
ax.set_ylabel("Nombre de parells", fontsize=12)
ax.set_title("Distribució de similitud: parells equivalents vs no equivalents")
ax.legend(fontsize=11)
ax.axvline(x=0.8, color="black", linestyle="--", alpha=0.5, label="Possible llindar")
plt.tight_layout()
plt.show()

print(f"Similitud mitjana — Equivalents: {equiv.mean():.3f}")
print(f"Similitud mitjana — No equivalents: {no_equiv.mean():.3f}")

In [ ]:
# Podem usar un llindar per "classificar" parells com a equivalents?
llindar = 0.8

prediccions = (df["similitud"] >= llindar).astype(int)
encerts = (prediccions == df["label"]).mean()

print(f"Llindar: {llindar}")
print(f"Accuracy: {encerts:.1%}")
print()

# Analitzem els errors
fp = ((prediccions == 1) & (df["label"] == 0)).sum()  # Prediu equivalent, però no ho és
fn = ((prediccions == 0) & (df["label"] == 1)).sum()  # Prediu no equivalent, però sí ho és
tp = ((prediccions == 1) & (df["label"] == 1)).sum()
tn = ((prediccions == 0) & (df["label"] == 0)).sum()

print("Matriu de confusió:")
print(f"  TP (equivalent → equivalent):       {tp}")
print(f"  TN (no equivalent → no equivalent): {tn}")
print(f"  FP (no equivalent → equivalent):    {fp}   ⚠️ El model creu que són iguals, però no")
print(f"  FN (equivalent → no equivalent):    {fn}  ⚠️ El model no veu que són iguals")
print()

# Provem diversos llindars
print(f"{'Llindar':>8} | {'Accuracy':>8} | {'FP':>5} | {'FN':>5} | {'Precisió':>8} | {'Exhaustivitat (recall)':>8}")
print("-" * 60)
for l in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9]:
    pred = (df["similitud"] >= l).astype(int)
    acc = (pred == df["label"]).mean()
    fp_l = ((pred == 1) & (df["label"] == 0)).sum()
    fn_l = ((pred == 0) & (df["label"] == 1)).sum()
    tp_l = ((pred == 1) & (df["label"] == 1)).sum()
    prec = tp_l / (tp_l + fp_l) if (tp_l + fp_l) > 0 else 0
    rec = tp_l / (tp_l + fn_l) if (tp_l + fn_l) > 0 else 0
    print(f"  {l:.2f}   |  {acc:.1%}   | {fp_l:>5} | {fn_l:>5} |  {prec:.1%}   |  {rec:.1%}")

### Reflexió

Amb un simple càlcul de similitud cosinus (sense entrenar res!) podem classificar parells de preguntes amb una accuracy sorprenent. Això demostra el poder dels embeddings pre-entrenats.

# 📝 Exercicis

## Exercici 1 — Similitud entre especialitats

Crea una llista de 12 frases clíniques (4 de cardiologia, 4 de neurologia, 4 de traumatologia) i genera la matriu de similitud.

- Es formen 3 grups visuals?
- Quines especialitats queden "més a prop" entre elles? Per què pot ser?

In [ ]:
# El teu codi aquí

## Exercici 2 — Sinònims mèdics

Calcula la similitud cosinus entre els parells següents. Quins parells detecta bé el model? Algun li costa?

```python
parells = [
    ("heart attack", "myocardial infarction"),
    ("high blood pressure", "hypertension"),
    ("broken bone", "fracture"),
    ("stomach ache", "abdominal pain"),
    ("runny nose", "rhinorrhea"),
    ("stroke", "cerebrovascular accident"),
    ("sugar disease", "diabetes mellitus"),
    ("feeling blue", "major depressive disorder"),
]
```

In [ ]:
# El teu codi aquí

## Exercici 3 — Cercador amb filtre de similitud mínima

Modifica la funció `cercar()` perquè accepti un paràmetre `min_sim` i només retorni resultats amb similitud per sobre d'aquest llindar.

Prova-ho amb `min_sim=0.5` i amb una query no mèdica. Quants resultats retorna?

In [ ]:
# El teu codi aquí

## Exercici 4 — Explora els errors

Del dataset `medical_questions_pairs`, troba:

**A)** Els 5 parells etiquetats com a **equivalents** amb la similitud **més baixa** (el model s'equivoca més).

**B)** Els 5 parells etiquetats com a **no equivalents** amb la similitud **més alta** (el model els confon).

Mira les preguntes: entens per què el model s'equivoca? Potser en algun cas l'etiqueta és discutible?

In [ ]:
# El teu codi aquí

## Exercici 5 — Classificador zero-shot amb embeddings

Sense entrenar cap model, podem classificar preguntes per tema. La idea:

1. Defineix categories amb una frase descriptiva:
```python
categories = {
    "cardiology": "heart disease, chest pain, blood pressure, cardiac problems",
    "neurology": "headache, brain, seizures, neurological symptoms",
    "dermatology": "skin rash, itching, acne, skin problems",
    "mental_health": "anxiety, depression, stress, psychological problems",
    "gastroenterology": "stomach pain, digestion, nausea, bowel problems",
}
```

2. Genera l'embedding de cada categoria
3. Per cada pregunta, calcula la similitud amb cada categoria
4. Assigna la categoria amb més similitud

Prova-ho amb 20 preguntes del corpus. Té sentit la classificació?

In [ ]:
# El teu codi aquí

## Exercici 6 — Cercador semàntic millorat

Millora el cercador de la Part 4:

**A)** Afegeix un **resum dels resultats**: compta quantes vegades apareix cada paraula clau als resultats (sense stopwords) i mostra les 5 més freqüents.

**B)** Implementa una funció `cercar_similar(pregunta_index)` que, donada una pregunta del corpus pel seu índex, trobi les 5 més similars (excloent-se a si mateixa). Això és útil per fer recomanacions: "preguntes relacionades".

**C)** Afegeix la capacitat de fer cerques **negatives**: donada una query i una paraula a excloure, filtra resultats. Per exemple: cercar "pain" excloent resultats sobre "headache".

In [ ]:
# El teu codi aquí